In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [2]:
import gradio as gr
from Day4LLMCalling import Llms, tools
from openai import OpenAI
from google import genai

client = genai.Client(api_key=os.getenv('GOOGLE_API_KEY'))
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=os.getenv('GOOGLE_API_KEY'))


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
import base64
from PIL import Image
from io import BytesIO

def artist(city):
    if not city:
        return None
    image_response = gemini.images.generate(
        prompt=f'Generate an artistic image for {city}',
        model="gemini-3.1-flash-image",
        n=1,
        size = "1024x1024",
        response_format="b64_json"
    )
    image_b64 = base64.b64decode(image_response.data[0].b64_json)
    image_result = Image.open(BytesIO(image_b64))
    return image_result



In [4]:
def talker(text):
    # Using the native Gemini 3.1 Flash TTS model
    response = client.models.generate_content(
        model="gemini-3.1-flash-tts-preview",
        contents=text
    )
    
    # The response contains the raw audio bytes
    audio_bytes = response.executable_ad_data # or response.data depending on version
    return audio_bytes

In [5]:
from Day4LLMCalling.messageSeries import mSeries
import openai.types.chat.chat_completion_message as msg

def wrapLlm(message):
    response,tool_arguments = Llms.callModel(message, source='gemini',tools=tools,return_tool_arguments=True)
    history = mSeries.promptList.get(0,{}).get('gemini-3-flash-preview',[])
    for item in history:
        print(type(item))
        if isinstance(item, msg.ChatCompletionMessage):
            item = {'role': 'assistant', 'content': 'tool called'}
    print(history)
    if len(tool_arguments)==0:
        return response, history, None
    return response, history, tool_arguments[0]['destination_city']

In [6]:
with gr.Blocks() as ui:
    city_state = gr.State()
    audio_state = gr.State()
    with gr.Row():
        chat_history = gr.Chatbot(height=500, label='Chat History')
        image_box = gr.Image(height=500, interactive=False, show_label=False)
    with gr.Row():
        audio_box = gr.Audio(autoplay=True)
    with gr.Row():
        message_box = gr.Textbox(label='Chat with AI')

        message_box.submit(
            fn = wrapLlm,
            inputs=[message_box],
            outputs=[audio_state,chat_history,city_state]
            ).then(
            fn=artist,
            inputs=[city_state], 
            outputs=[image_box])
            # ).then(
            # fn=talker,
            # inputs=[audio_state],
            # outputs=[audio_box]
            # )

ui.launch()
        

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


<class 'dict'>
<class 'dict'>
<class 'dict'>
[{'role': 'system', 'content': ''}, {'role': 'user', 'content': 'hi there'}, {'role': 'assistant', 'content': "Hello! How can I help you today? Whether you're looking for flight prices or have other questions, feel free to ask."}]
called get_by_name on london


Traceback (most recent call last):
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\queueing.py", line 785, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 2172, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\gradio\blocks.py", line 1634, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\PGCP-AI\projects\MyLLMLearning\.venv\Lib\site-packages\an

In [7]:
mSeries.promptList

{0: {'gemini-3-flash-preview': [{'role': 'system', 'content': ''},
   {'role': 'user', 'content': 'hi there'},
   {'role': 'assistant',
    'content': "Hello! How can I help you today? Whether you're looking for flight prices or have other questions, feel free to ask."},
   {'role': 'user', 'content': "What's the price for london?"},
   ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='098uiwvb', function=Function(arguments='{"action":"get_by_name","destination_city":"London"}', name='set_ticket_price'), type='function', extra_content={'google': {'thought_signature': 'Ev4CCvsCAQw51sdSxd32aehhOBt5ycDR/E+xMLJ9n0+RcNtCkkelEQ0Jj2nrjxhGyPb7Oqxh1ZDRV0aJa7JxeDiLGF7j4vTuFwX78SHKCYUvlR7f3TGMPy7Fi7/dQrjAz/Q2BqhK+JyzyM2+gCZ0ImATRMwF7PhhtKD2kK1h5F1SVgQu54lP5PS5yD+Pc3BhlU897iSEP4YWfwRyx31NGm3FwWxlDCdeP+eAuEJFMtvo8YAxW7HKU1a69KVg9srFQ09zzL5nmhsKYi3GUR1oM62+wsyu5mINGG5/nh1PpgIZADD